In [7]:
import numpy as np
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

def load_cifar10(batch_size=16, sample_ratio=1.0, seed=42, data_root="./data"):
    """
    加载CIFAR-10数据集（支持采样）

    Args:
        batch_size: 批次大小
        sample_ratio: 采样比例（默认1.0=100%，比赛要求≤10%）
        seed: 随机种子，保证可复现
        data_root: 数据集根目录（默认 ./data 当前目录下的data文件夹）
    """
    # ✅ 1. 设置随机种子，保证可复现
    np.random.seed(seed)
    torch.manual_seed(seed)

    # ✅ 2. 数据增强/预处理
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        # ✅ 3. 添加归一化（提升模型稳定性）
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    # ✅ 4. 加载数据集
    test_dataset = datasets.CIFAR10(
        root=data_root,
        train=False,
        download=True,
        transform=transform
    )

    total_len = len(test_dataset)
    sample_len = int(total_len * sample_ratio)

    # ✅ 5. 采样
    indices = np.random.choice(total_len, sample_len, replace=False)
    subset = Subset(test_dataset, indices)

    # ✅ 6. 创建DataLoader
    test_loader = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,        # 多进程加载
        pin_memory=True       # 加速GPU传输
    )

    # ✅ 7. 打印采样信息（便于合规性检查）
    print(f"📊 CIFAR-10 数据加载完成:")
    print(f"  总样本数: {total_len}")
    print(f"  采样样本数: {sample_len} ({sample_ratio*100:.1f}%)")
    print(f"  批次大小: {batch_size}")
    print(f"  批次数: {len(test_loader)}")
    print(f"  数据使用比例: {sample_ratio*100:.1f}% (比赛要求≤10%)")

    # ✅ 8. 合规性检查
    if sample_ratio > 0.1:
        print(f"  ⚠️ 警告: 数据使用比例 {sample_ratio*100:.1f}% 超过10%限制！")

    return test_loader


def get_data_usage_ratio(sample_ratio):
    """
    获取数据使用比例（用于合规性检查）

    Args:
        sample_ratio: 采样比例

    Returns:
        float: 数据使用比例
    """
    return sample_ratio


def verify_data_compliance(sample_ratio):
    """
    验证数据使用比例是否符合比赛要求

    Args:
        sample_ratio: 采样比例

    Returns:
        dict: 合规性检查结果
    """
    return {
        'sample_ratio': sample_ratio,
        'usage_percentage': sample_ratio * 100,
        'is_compliant': sample_ratio <= 0.1,
        'limit': '≤10%'
    }


# ==========================================
# 测试代码
# ==========================================
if __name__ == "__main__":
    print("=" * 60)
    print("🧪 data_loader.py 测试")
    print("=" * 60)

    # 测试1: 加载数据（10%采样）
    print("\n[测试1] 加载CIFAR-10 (10%采样)")
    loader = load_cifar10(batch_size=16, sample_ratio=0.1)

    # 测试2: 获取一个batch
    print("\n[测试2] 获取一个batch")
    for images, labels in loader:
        print(f"  ✅ 图像张量形状: {images.shape}")
        print(f"  ✅ 标签: {labels[:5].tolist()}")
        print(f"  ✅ 图像值范围: [{images.min():.3f}, {images.max():.3f}]")
        break

    # 测试3: 合规性检查
    print("\n[测试3] 合规性检查")
    compliance = verify_data_compliance(0.1)
    print(f"  ✅ 数据使用比例: {compliance['usage_percentage']:.1f}%")
    print(f"  ✅ 合规状态: {'✅ 合规' if compliance['is_compliant'] else '❌ 不合规'}")

    # 测试4: 测试超过10%的情况
    print("\n[测试4] 测试超过10%限制（应警告）")
    loader_over = load_cifar10(batch_size=16, sample_ratio=0.5)

    print("\n" + "=" * 60)
    print("✅ data_loader.py 测试通过！")
    print("=" * 60)


🧪 data_loader.py 测试

[测试1] 加载CIFAR-10 (10%采样)
📊 CIFAR-10 数据加载完成:
  总样本数: 10000
  采样样本数: 1000 (10.0%)
  批次大小: 16
  批次数: 63
  数据使用比例: 10.0% (比赛要求≤10%)

[测试2] 获取一个batch
  ✅ 图像张量形状: torch.Size([16, 3, 64, 64])
  ✅ 标签: [2, 1, 5, 8, 9]
  ✅ 图像值范围: [-1.000, 1.000]

[测试3] 合规性检查
  ✅ 数据使用比例: 10.0%
  ✅ 合规状态: ✅ 合规

[测试4] 测试超过10%限制（应警告）
📊 CIFAR-10 数据加载完成:
  总样本数: 10000
  采样样本数: 5000 (50.0%)
  批次大小: 16
  批次数: 313
  数据使用比例: 50.0% (比赛要求≤10%)
  ⚠️ 警告: 数据使用比例 50.0% 超过10%限制！

✅ data_loader.py 测试通过！
